<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-05-llm-api-engineering/lesson-5.2-streaming-and-realtime/notebooks/exercises-5_2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 5.2: Streaming & Real-Time Patterns

*Module 5 · Building Production Applications*

> Without streaming, users stare at a blank screen for 5 seconds. With streaming, the first token appears in 200ms. This lesson builds streaming pipelines: SDK streaming from all providers, FastAPI SSE endpoints, WebSocket bridges, and Cloud Run deployment with anti-buffering config.

`SSE` · `FastAPI` · `WebSockets` · `Cloud Run` · `60 min`

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-5.2.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — SDK Streaming — All Three Providers — `01_sdk_streaming.py`
2. Step 2 — FastAPI SSE Endpoint — Stream to Frontend — `02_fastapi_sse.py`
3. Step 3 — WebSocket Bridge — Full-Duplex Chat — `03_websocket.py`
4. Step 4 — Deploy to Cloud Run — Production Streaming — `04_cloud_run_deploy.sh`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q anthropic>=0.40.0 google-generativeai openai fastapi


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export ANTHROPIC_API_KEY=sk-...
#   export GOOGLE_API_KEY=sk-...
#   export OPENAI_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("ANTHROPIC_API_KEY", "YOUR_ANTHROPIC_API_KEY_HERE")
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")
os.environ.setdefault("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · SDK Streaming — All Three Providers

Stream tokens from Gemini, OpenAI, Claude. Measure TTFT for each.

**`01_sdk_streaming.py`** · _Block 1 of 4_

SDK STREAMING — Gemini, OpenAI, Claude


In [ ]:
# SDK STREAMING — Gemini, OpenAI, Claude
import os, time

prompt = "Explain streaming in LLM APIs in 3 sentences."

# Gemini
import google.generativeai as genai
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")
start = time.time(); ttft = None
for chunk in model.generate_content(prompt, stream=True):
    if ttft is None: ttft = time.time()-start
    print(chunk.text, end="", flush=True)
print(f"\nGemini TTFT: {ttft*1000:.0f}ms\n")

# OpenAI
from openai import OpenAI
oai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
start = time.time(); ttft = None
for chunk in oai.chat.completions.create(
    model="gpt-4o", messages=[{"role":"user","content":prompt}], stream=True):
    delta = chunk.choices[0].delta.content
    if delta:
        if ttft is None: ttft = time.time()-start
        print(delta, end="", flush=True)
print(f"\nOpenAI TTFT: {ttft*1000:.0f}ms\n")

# Claude
import anthropic
cl = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
start = time.time(); ttft = None
with cl.messages.stream(model="claude-sonnet-4-20250514",max_tokens=200,
    messages=[{"role":"user","content":prompt}]) as s:
    for t in s.text_stream:
        if ttft is None: ttft = time.time()-start
        print(t, end="", flush=True)
print(f"\nClaude TTFT: {ttft*1000:.0f}ms")


> **What just happened?** Three providers, three streaming APIs. Gemini: stream=True . OpenAI: stream=True . Claude: messages.stream() . All yield chunks with TTFT ~200-400ms. TTFT is the key UX metric. Users start reading immediately instead of waiting for the full response.


### Step 2 · FastAPI SSE Endpoint — Stream to Frontend

Build an API that streams LLM tokens via Server-Sent Events.

**`02_fastapi_sse.py`** · _Block 2 of 4_

FASTAPI SSE — Stream LLM to frontend


In [ ]:
# FASTAPI SSE — Stream LLM to frontend
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
import google.generativeai as genai
import json, os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

async def stream_llm(prompt):
    model = genai.GenerativeModel("gemini-2.0-flash")
    for chunk in model.generate_content(prompt, stream=True):
        if chunk.text:
            yield f"data: {json.dumps({'token': chunk.text})}\n\n"
    yield "data: [DONE]\n\n"

@app.post("/chat/stream")
async def chat_stream(request: Request):
    body = await request.json()
    return StreamingResponse(stream_llm(body["prompt"]),
        media_type="text/event-stream",
        headers={"Cache-Control":"no-cache", "X-Accel-Buffering":"no"})
# Critical: X-Accel-Buffering:no prevents nginx/Cloud Run buffering


### Step 3 · WebSocket Bridge — Full-Duplex Chat

Client can send AND receive simultaneously. User can interrupt mid-generation.

**`03_websocket.py`** · _Block 3 of 4_

WEBSOCKET BRIDGE — Full-duplex LLM streaming


In [ ]:
# WEBSOCKET BRIDGE — Full-duplex LLM streaming
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
import google.generativeai as genai
import json, os, asyncio

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
app = FastAPI()

@app.websocket("/ws/chat")
async def ws_chat(ws: WebSocket):
    await ws.accept()
    model = genai.GenerativeModel("gemini-2.0-flash")
    try:
        while True:
            data = json.loads(await ws.receive_text())
            await ws.send_json({"type": "start"})
            for chunk in model.generate_content(data["prompt"], stream=True):
                if chunk.text:
                    await ws.send_json({"type":"token", "text":chunk.text})
                    await asyncio.sleep(0)
            await ws.send_json({"type": "done"})
    except WebSocketDisconnect: pass
# Advantage: client can send NEW message while AI is still responding


### Step 4 · Deploy to Cloud Run — Production Streaming

Auto-scaling, HTTPS, global CDN. Critical flags for streaming to work.

**`04_cloud_run_deploy.sh`** · _Block 4 of 4_

Cloud Run deployment for streaming API


In [ ]:
# Cloud Run deployment for streaming API
gcloud run deploy genai-streaming \
  --source . \
  --region asia-south1 \
  --allow-unauthenticated \
  --memory 512Mi --cpu 1 \
  --min-instances 0 --max-instances 10 \
  --timeout 300 \
  --no-cpu-throttling \
  --set-env-vars GOOGLE_API_KEY=$GOOGLE_API_KEY

# Critical flags:
# --no-cpu-throttling  Keep CPU active during streaming
# --timeout 300        Allow long responses to complete
# X-Accel-Buffering:no  In FastAPI headers (prevents buffer)
# Cost: ~$0 at low traffic (scales to zero)


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
